[![In Colab öffnen](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Y-Robin/DeepLearningKurs/blob/main/notebooks/02_tag_1_reinforcement_learning_mini_pong.ipynb)

# Tag 1 Reinforcement Learning Mini Pong

Dieses Notebook zeigt ein sehr kleines Pong-ähnliches Reinforcement-Learning-Beispiel ohne Gym, Atari oder GPU. Es erklärt Agent, Environment, State, Action, Reward, Policy und Episode anhand einer eigenen Grid-Umgebung.

## 1. Grundbegriffe

- **Agent**: trifft Entscheidungen, hier bewegt er das Paddle.
- **Environment**: die Spielwelt mit Ball, Paddle und Regeln.
- **State**: kompakte Beschreibung der aktuellen Situation.
- **Action**: mögliche Handlung: links, stehen oder rechts.
- **Reward**: Rückmeldung der Umgebung, z. B. `+1` bei Treffer und `-1` bei Verpassen.
- **Policy**: Strategie, nach der der Agent Aktionen auswählt.
- **Episode**: ein Spielversuch bis Miss oder maximale Schrittzahl.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import clear_output
import ipywidgets as widgets

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
plt.rcParams['figure.figsize'] = (6, 4)
ACTIONS = [-1, 0, 1]  # links, stehen, rechts
ACTION_NAMES = {-1: 'links', 0: 'stehen', 1: 'rechts'}
print('Setup abgeschlossen.')

## 2. Minimalistische Grid-Umgebung

Der Ball bewegt sich diagonal im Grid. Unten liegt ein Paddle mit Breite 3. Trifft der Ball das Paddle, gibt es `+1`. Verpasst der Agent den Ball, gibt es `-1`. Eine kleine Schrittstrafe kann sparsames Spielen fördern.

In [ ]:
class MiniPongEnv:
    def __init__(self, width=7, height=6, max_steps=60, step_penalty=-0.01, seed=RANDOM_STATE):
        self.width = int(width)
        self.height = int(height)
        self.max_steps = int(max_steps)
        self.step_penalty = float(step_penalty)
        self.rng = np.random.default_rng(seed)
        self.paddle_width = 3
        self.reset()

    def reset(self):
        self.ball_x = int(self.rng.integers(1, self.width - 1))
        self.ball_y = 0
        self.vx = int(self.rng.choice([-1, 1]))
        self.vy = 1
        self.paddle_x = self.width // 2
        self.steps = 0
        return self.state()

    def state(self):
        # Diskreter Zustand für die Q-Tabelle.
        return (self.ball_x, self.ball_y, self.vx, self.paddle_x)

    def step(self, action):
        action = int(action)
        self.paddle_x = int(np.clip(self.paddle_x + action, 1, self.width - 2))
        self.ball_x += self.vx
        self.ball_y += self.vy
        self.steps += 1

        if self.ball_x <= 0 or self.ball_x >= self.width - 1:
            self.vx *= -1
            self.ball_x = int(np.clip(self.ball_x, 0, self.width - 1))

        reward = self.step_penalty
        done = False
        if self.ball_y >= self.height - 1:
            hit = abs(self.ball_x - self.paddle_x) <= self.paddle_width // 2
            reward = 1.0 if hit else -1.0
            done = True
        if self.steps >= self.max_steps:
            done = True
        return self.state(), reward, done, {'hit': reward > 0}

    def render_array(self):
        grid = np.zeros((self.height, self.width))
        grid[self.ball_y, self.ball_x] = 2
        for x in range(self.paddle_x - self.paddle_width // 2, self.paddle_x + self.paddle_width // 2 + 1):
            if 0 <= x < self.width:
                grid[self.height - 1, x] = 1
        return grid

def plot_env(env, title='Mini Pong'):
    plt.imshow(env.render_array(), cmap='viridis', vmin=0, vmax=2)
    plt.xticks(range(env.width)); plt.yticks(range(env.height))
    plt.title(title)
    plt.show()

In [ ]:
env = MiniPongEnv(width=7, seed=RANDOM_STATE)
plot_env(env, 'Startzustand')
print('State:', env.state())
print('Aktionen:', ACTION_NAMES)

## 3. Random Agent

Ein Random Agent wählt zufällig links, stehen oder rechts. Das ist unsere einfache Basislinie.

In [ ]:
def random_policy(state, rng):
    return int(rng.choice(ACTIONS))

def run_episode(env, policy, max_steps=60, render=False, seed=RANDOM_STATE):
    rng = np.random.default_rng(seed)
    state = env.reset()
    total_reward = 0.0
    frames = []
    for _ in range(max_steps):
        action = policy(state, rng)
        state, reward, done, info = env.step(action)
        total_reward += reward
        if render:
            frames.append(env.render_array().copy())
        if done:
            break
    return total_reward, frames

random_rewards = [run_episode(MiniPongEnv(seed=RANDOM_STATE+i), random_policy, seed=RANDOM_STATE+i)[0] for i in range(100)]
print('Random Agent: mittlerer Reward über 100 Episoden =', round(np.mean(random_rewards), 3))

## 4. Q-Learning mit Q-Tabelle

Die Q-Tabelle speichert für jeden Zustand und jede Aktion einen geschätzten Wert. Die Update-Regel lernt aus Reward plus geschätztem Zukunftswert.

In [ ]:
def choose_action(q_table, state, epsilon, rng):
    if rng.random() < epsilon or state not in q_table:
        return int(rng.choice(ACTIONS))
    return ACTIONS[int(np.argmax(q_table[state]))]

def train_q_learning(alpha=0.3, gamma=0.9, epsilon=0.2, episodes=500, width=7, seed=RANDOM_STATE):
    rng = np.random.default_rng(seed)
    q_table = {}
    rewards = []
    for episode in range(int(episodes)):
        env = MiniPongEnv(width=int(width), seed=seed + episode)
        state = env.reset()
        total_reward = 0.0
        done = False
        while not done:
            q_table.setdefault(state, np.zeros(len(ACTIONS)))
            action = choose_action(q_table, state, epsilon, rng)
            next_state, reward, done, info = env.step(action)
            q_table.setdefault(next_state, np.zeros(len(ACTIONS)))
            a_idx = ACTIONS.index(action)
            target = reward + gamma * np.max(q_table[next_state]) * (not done)
            q_table[state][a_idx] += alpha * (target - q_table[state][a_idx])
            state = next_state
            total_reward += reward
        rewards.append(total_reward)
    return q_table, np.array(rewards)

def q_policy_from_table(q_table):
    def policy(state, rng):
        if state not in q_table:
            return 0
        return ACTIONS[int(np.argmax(q_table[state]))]
    return policy

q_table, rewards = train_q_learning()
plt.plot(pd.Series(rewards).rolling(25, min_periods=1).mean())
plt.xlabel('Episode')
plt.ylabel('Reward, gleitender Mittelwert')
plt.title('Trainingskurve Q-Learning')
plt.show()

In [ ]:
def evaluate(policy, episodes=100, width=7):
    rewards = []
    for i in range(episodes):
        reward, _ = run_episode(MiniPongEnv(width=width, seed=RANDOM_STATE + i), policy, seed=RANDOM_STATE + i)
        rewards.append(reward)
    return float(np.mean(rewards)), float(np.std(rewards))

before_mean, before_std = evaluate(random_policy)
after_mean, after_std = evaluate(q_policy_from_table(q_table))
print(f'Vor Training / Random: Mittelwert={before_mean:.3f}, Std={before_std:.3f}')
print(f'Nach Training / Q-Policy: Mittelwert={after_mean:.3f}, Std={after_std:.3f}')

## 5. Slider: Hyperparameter live verändern

Die Slider zeigen, dass Lernrate, Discount, Epsilon, Episodenanzahl und Spielfeldbreite die Lernkurve verändern.

In [ ]:
def q_learning_widget(alpha=0.3, gamma=0.9, epsilon=0.2, episodenanzahl=500, spielfeldbreite=7):
    q, r = train_q_learning(alpha=alpha, gamma=gamma, epsilon=epsilon, episodes=episodenanzahl, width=spielfeldbreite)
    mean_eval, std_eval = evaluate(q_policy_from_table(q), episodes=80, width=spielfeldbreite)
    plt.plot(pd.Series(r).rolling(25, min_periods=1).mean())
    plt.xlabel('Episode')
    plt.ylabel('Reward')
    plt.title(f'Evaluation: {mean_eval:.2f} ± {std_eval:.2f}')
    plt.show()

widgets.interact(
    q_learning_widget,
    alpha=widgets.FloatSlider(value=0.3, min=0.05, max=0.9, step=0.05),
    gamma=widgets.FloatSlider(value=0.9, min=0.1, max=0.99, step=0.05),
    epsilon=widgets.FloatSlider(value=0.2, min=0.0, max=0.8, step=0.05),
    episodenanzahl=widgets.IntSlider(value=500, min=100, max=1500, step=100),
    spielfeldbreite=widgets.IntSlider(value=7, min=5, max=11, step=2),
);

## 6. Episode visualisieren

Die folgende Zelle zeigt eine gelernte Episode Schritt für Schritt. In VS Code, Jupyter und Colab kann die Ausgabe wiederholt aktualisiert werden.

In [ ]:
def visualize_episode(q_table, width=7, pause=0.25):
    env = MiniPongEnv(width=width, seed=RANDOM_STATE + 999)
    policy = q_policy_from_table(q_table)
    state = env.reset()
    total_reward = 0
    for step in range(env.max_steps):
        clear_output(wait=True)
        plot_env(env, title=f'Schritt {step}, State={state}')
        action = policy(state, np.random.default_rng(RANDOM_STATE))
        state, reward, done, info = env.step(action)
        total_reward += reward
        plt.pause(pause)
        if done:
            clear_output(wait=True)
            plot_env(env, title=f'Ende nach {step+1} Schritten, Reward={total_reward:.2f}')
            break

visualize_episode(q_table, width=7, pause=0.15)

## 7. Reflexion

Warum ist echtes Pong deutlich schwerer?

- Der Zustand besteht aus Pixeln statt aus wenigen diskreten Zahlen.
- Es gibt längere Episoden und verzögerte Rewards.
- Die Dynamik ist schneller und kontinuierlicher.
- Große neuronale Netze, Replay Buffer und sorgfältiges Tuning werden typischerweise nötig.

**Reflexionsfrage:** Welche Information aus dem echten Spielbild müsste ein Agent erst lernen, bevor er gute Aktionen wählen kann?